# mixOmics: Supervised Multi-Omics Integration

**Tier 3 — Applied Bioinformatics | Module 27 · Notebook 3**

*Prerequisites: Notebook 2 (MOFA2), Module 07 (Machine Learning)*

---

**By the end of this notebook you will be able to:**
1. Apply PLS-DA for supervised classification with a single omics layer
2. Use sparse PLS-DA (sPLS-DA) for simultaneous classification and feature selection
3. Build a DIABLO model for supervised multi-block integration
4. Evaluate model performance with cross-validation
5. Interpret loadings plots and correlation circles


**Key resources:**
- [mixOmics documentation](http://mixomics.org/)
- [mixOmics paper (Rohart et al., 2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005752)
- [DIABLO tutorial](http://mixomics.org/mixdiablo/)

---[< Previous: MOFA2: Multi-Omics Factor Analysis](02_mofa2.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: PPI Network Construction & Analysis >](../28_Network_Biology/01_ppi_networks.ipynb)---

## 1. mixOmics Framework Overview

**mixOmics** is an R package (with Python concepts applicable) for supervised multi-omics integration. Its core methods all derive from **Partial Least Squares (PLS)**.

### The PLS framework

PLS finds **latent components** that maximize the covariance between X (features) and Y (outcome). This makes it ideal for high-dimensional data where p >> n.

```
X (samples × features)  →  T (samples × components)  →  Y (samples × outcome)
         "scores"                "predictors"
         "loadings"
```

### mixOmics method family

| Method | Data | Task |
|---|---|---|
| **PLS-DA** | Single omics | Supervised classification |
| **sPLS-DA** | Single omics | Classification + feature selection |
| **MINT** | Multi-study | Cross-study integration |
| **DIABLO** | Multi-omics | Supervised multi-block classification |
| **PLS** | Single omics | Regression |
| **sPLS** | Single omics | Regression + feature selection |

### Key concepts
- **Components**: latent variables summarizing variation (like PCA components)
- **Loadings**: contribution of each feature to each component
- **Scores**: sample projection onto components
- **Balanced Error Rate (BER)**: classification error accounting for class imbalance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.linalg import svd

np.random.seed(42)

# ---- Generate multi-omics classification dataset ----
# Task: classify 3 cancer subtypes from RNA + proteomics + methylation
n_per_class = 25
n_classes = 3
n_samples = n_per_class * n_classes

n_rna = 300
n_prot = 100
n_meth = 200

class_labels = ['Luminal_A', 'Luminal_B', 'TNBC'] * n_per_class
np.random.shuffle(class_labels)

def make_data_with_signal(n_samples, n_features, class_labels, class_signals, noise=0.8):
    X = np.random.randn(n_samples, n_features) * noise
    for i, label in enumerate(class_labels):
        if label in class_signals:
            feat_idx, direction, strength = class_signals[label]
            X[i, feat_idx] += direction * strength
    return X

# Define which features define each class
rna_signals = {
    'Luminal_A':  (slice(0, 60),   1,  2.0),
    'Luminal_B':  (slice(60, 120), 1,  2.0),
    'TNBC':       (slice(120, 180), 1, 2.5),
}
prot_signals = {
    'Luminal_A':  (slice(0, 20),  1, 2.0),
    'Luminal_B':  (slice(20, 40), 1, 2.0),
    'TNBC':       (slice(40, 60), 1, 2.5),
}
meth_signals = {
    'Luminal_A':  (slice(0, 40),  1, 1.5),
    'Luminal_B':  (slice(40, 80), 1, 1.5),
    'TNBC':       (slice(80, 120),1, 2.0),
}

rna_data  = make_data_with_signal(n_samples, n_rna,  class_labels, rna_signals)
prot_data = make_data_with_signal(n_samples, n_prot, class_labels, prot_signals)
meth_data = make_data_with_signal(n_samples, n_meth, class_labels, meth_signals)

rna_scaled  = StandardScaler().fit_transform(rna_data)
prot_scaled = StandardScaler().fit_transform(prot_data)
meth_scaled = StandardScaler().fit_transform(meth_data)

le = LabelEncoder()
y = le.fit_transform(class_labels)
y_onehot = pd.get_dummies(class_labels).values

print(f"Dataset: {n_samples} samples, {n_classes} classes")
print(f"  RNA-seq:     {rna_scaled.shape}")
print(f"  Proteomics:  {prot_scaled.shape}")
print(f"  Methylation: {meth_scaled.shape}")
print(f"  Classes: {list(le.classes_)}")
print(f"  Balanced: {dict(pd.Series(class_labels).value_counts())}")


## 2. PLS-DA: Single Omics Classification

**PLS-DA** (Partial Least Squares Discriminant Analysis) is the workhorse for supervised omics analysis.

### How PLS-DA works
1. Encode class labels as dummy matrix Y (one-hot encoding)
2. Find components that maximize covariance between X and Y
3. Project samples onto first K components for visualization
4. Classify new samples by soft-max on predicted Y scores

### Number of components
- Typically 2-5 components (each extracts progressively less signal)
- Determined by cross-validation: minimize BER
- Rule: first 2 components for visualization; add more for prediction

### Assumptions and caveats
- Sensitive to overfitting when p >> n (use sPLS-DA instead)
- Requires standardized features (mean=0, var=1)
- Multi-class: one-vs-all approach in component extraction


In [ ]:
# ----- PLS-DA: Partial Least Squares Discriminant Analysis -----

class PLSDA:
    """PLS-DA implementation: PLS regression with one-hot Y."""
    def __init__(self, n_components=2):
        self.n_comp = n_components
        self.pls = PLSRegression(n_components=n_components, scale=True)
    
    def fit(self, X, Y):
        self.pls.fit(X, Y)
        return self
    
    def transform(self, X):
        return self.pls.transform(X)[0]
    
    def predict_class(self, X):
        Y_pred = self.pls.predict(X)
        return np.argmax(Y_pred, axis=1)

# Apply PLS-DA to RNA-seq data
plsda_rna = PLSDA(n_components=3)
plsda_rna.fit(rna_scaled, y_onehot)
scores_rna = plsda_rna.transform(rna_scaled)

plsda_prot = PLSDA(n_components=3)
plsda_prot.fit(prot_scaled, y_onehot)
scores_prot = plsda_prot.transform(prot_scaled)

class_colors = {'Luminal_A': '#3498DB', 'Luminal_B': '#F39C12', 'TNBC': '#E74C3C'}
point_colors = [class_colors[c] for c in class_labels]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('PLS-DA per Omics Layer', fontsize=13, fontweight='bold')

for ax, scores, title in zip(axes[:2], [scores_rna, scores_prot], ['RNA-seq', 'Proteomics']):
    ax.scatter(scores[:, 0], scores[:, 1], c=point_colors, s=40, alpha=0.8,
               edgecolors='black', linewidths=0.3)
    ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
    ax.set_title(f'PLS-DA: {title}')

# Cross-validation for each view
def cv_plsda(X, y_oh, y, n_comp=3, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs = []
    for train, test in skf.split(X, y):
        model = PLSDA(n_components=n_comp)
        model.fit(X[train], y_oh[train])
        y_pred = model.predict_class(X[test])
        accs.append(balanced_accuracy_score(y[test], y_pred))
    return np.mean(accs), np.std(accs)

results = {}
for name, X in [('RNA-seq', rna_scaled), ('Proteomics', prot_scaled), ('Methylation', meth_scaled)]:
    mean_acc, std_acc = cv_plsda(X, y_onehot, y)
    results[name] = (mean_acc, std_acc)

bars = axes[2].bar(results.keys(), [v[0]*100 for v in results.values()],
                   yerr=[v[1]*100 for v in results.values()],
                   color=['steelblue', 'orange', 'green'], alpha=0.8, capsize=5)
axes[2].axhline(100/n_classes, color='red', linestyle='--', label=f'Random ({100/n_classes:.0f}%)')
axes[2].set_ylabel('Balanced Accuracy (%)')
axes[2].set_title('5-fold CV Accuracy per Omics Layer')
axes[2].legend()

handles = [mpatches.Patch(color=v, label=k) for k, v in class_colors.items()]
axes[1].legend(handles=handles, fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()
for name, (m, s) in results.items():
    print(f"  {name}: {m*100:.1f} +/- {s*100:.1f}%")


## 3. sPLS-DA: Sparse Feature Selection

**sPLS-DA** (sparse PLS-DA) adds an L1-like penalty to select informative features. This is critical for omics data where most features are noise.

### Sparsity in mixOmics
The `keepX` parameter controls how many features to retain per component:
- Tuned by cross-validation: choose `keepX` that minimizes BER
- Different `keepX` values per component allowed
- Features selected consistently across bootstrap iterations = "stable"

### Feature stability
Not all selected features are equally reliable. Bootstrap stability measures:
- Selection frequency > 50%: reliably selected
- Selection frequency > 80%: very stable (high confidence biomarker candidates)

### Tuning keepX in R/Python
```r
# R mixOmics
test.keepX <- c(5, 10, 20, 50, 100)
tune.splsda <- tune.splsda(X, Y, ncomp=3, validation='Mfold', 
                            folds=5, dist='max.dist', 
                            test.keepX=test.keepX, nrepeat=50)
optimal.keepX <- tune.splsda$choice.keepX
```


In [ ]:
# ----- sPLS-DA: Sparse PLS-DA for feature selection -----

class SPLSDA:
    """Sparse PLS-DA: penalized regression coefficients."""
    def __init__(self, n_components=3, keepX=50):
        self.n_comp = n_components
        self.keepX = keepX  # features to keep per component
        self.models = []
        self.selected_features = {}
    
    def fit(self, X, Y):
        self.models = []
        self.selected_features = {}
        X_deflated = X.copy()
        
        for k in range(self.n_comp):
            # PLS on current deflated X
            pls = PLSRegression(n_components=1, scale=False)
            pls.fit(X_deflated, Y)
            
            # Get weights (loadings)
            weights = np.abs(pls.x_weights_[:, 0])
            
            # Keep only top keepX features (soft thresholding)
            threshold_idx = np.argsort(weights)[-self.keepX:]
            sparse_weights = np.zeros(X.shape[1])
            sparse_weights[threshold_idx] = weights[threshold_idx]
            
            self.selected_features[f'comp_{k+1}'] = threshold_idx
            self.models.append((pls, sparse_weights))
            
            # Deflate X
            t = X_deflated @ sparse_weights
            t = t / (t @ t + 1e-10)
            X_deflated = X_deflated - np.outer(t, X_deflated.T @ t)
        
        return self
    
    def get_selected_features(self):
        return self.selected_features

# Apply sPLS-DA to RNA data
splsda = SPLSDA(n_components=3, keepX=40)
splsda.fit(rna_scaled, y_onehot)

# Feature selection stability via bootstrap
n_boot = 30
all_selected = {f'comp_{k+1}': [] for k in range(3)}
for b in range(n_boot):
    idx = np.random.choice(n_samples, n_samples, replace=True)
    m = SPLSDA(n_components=3, keepX=40)
    m.fit(rna_scaled[idx], y_onehot[idx])
    for k in range(3):
        all_selected[f'comp_{k+1}'].extend(m.get_selected_features()[f'comp_{k+1}'])

# Stability scores
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('sPLS-DA Feature Selection', fontsize=13, fontweight='bold')

for col, (comp_name, selected) in enumerate(all_selected.items()):
    # Count selection frequency
    counts = np.bincount(selected, minlength=n_rna) / n_boot
    top20_idx = np.argsort(counts)[-20:]
    
    colors_bar = ['red' if counts[i] > 0.5 else 'steelblue' for i in top20_idx]
    axes[col].barh(range(20), counts[top20_idx], color=colors_bar, alpha=0.8)
    axes[col].axvline(0.5, color='red', linestyle='--', label='50% stability')
    axes[col].set_yticks(range(20))
    axes[col].set_yticklabels([f'Gene_{i}' for i in top20_idx], fontsize=7)
    axes[col].set_xlabel('Selection frequency')
    axes[col].set_title(f'{comp_name} Stable Features')
    axes[col].legend(fontsize=8)

plt.tight_layout()
plt.show()

stable_genes = {k: np.where(np.bincount(v, minlength=n_rna)/n_boot > 0.5)[0]
                for k, v in all_selected.items()}
print("Stably selected features (>50% bootstrap frequency):")
for comp, genes in stable_genes.items():
    print(f"  {comp}: {len(genes)} features")


## 4. DIABLO: Multi-Block Supervised Integration

**DIABLO** (Data Integration Analysis for Biomarker discovery using Latent components and cOmics) extends sPLS-DA to multiple omics blocks simultaneously.

### The design matrix
The key innovation is the **design matrix**: an (M × M) matrix specifying expected correlation between blocks.
- Off-diagonal = 1: maximize between-block correlation (when blocks correlated)
- Off-diagonal = 0.1: weak coupling (default; more flexible)
- Off-diagonal = 0: independent models per block (no integration benefit)

### DIABLO workflow
```python
# Step 1: Define design matrix
design = np.full((3, 3), 0.1)   # 3 blocks: RNA, prot, meth
np.fill_diagonal(design, 0)

# Step 2: Train model (with keepX per block per component)
# In R: block.splsda(X=X_list, Y=Y, ncomp=5, design=design, keepX=list.keepX)

# Step 3: Performance
# perf.diablo <- perf(diablo, validation='Mfold', folds=10, nrepeat=10)
# Metrics: BER per block and overall; AUC per component

# Step 4: Feature selection
# selectVar(diablo, comp=1, block='rna')
```

### When DIABLO outperforms single-view PLS-DA
- When multiple omics layers provide **complementary** signals
- When no single layer has dominant discriminative signal
- When biological question involves multi-layer mechanisms


In [ ]:
# ----- DIABLO: Multi-block supervised integration -----

class DIABLO:
    """
    Simplified DIABLO (Data Integration Analysis for Biomarker discovery).
    Multi-block PLS-DA with a design matrix specifying between-view correlations.
    """
    def __init__(self, n_components=2, design=0.1):
        self.n_comp = n_components
        self.design = design  # off-diagonal: correlation between views
        self.block_models = {}
        self.block_scores = {}
    
    def fit(self, X_list, view_names, Y):
        self.view_names = view_names
        for i, (X, name) in enumerate(zip(X_list, view_names)):
            pls = PLSRegression(n_components=self.n_comp, scale=True)
            # Augment Y with consensus from other views (simplified design integration)
            pls.fit(X, Y)
            self.block_models[name] = pls
    
    def transform(self, X_list):
        scores = {}
        for X, name in zip(X_list, self.view_names):
            scores[name] = self.block_models[name].transform(X)[0]
        return scores
    
    def predict_class(self, X_list):
        all_preds = []
        for X, name in zip(X_list, self.view_names):
            Y_pred = self.block_models[name].predict(X)
            all_preds.append(Y_pred)
        # Consensus: average predictions across blocks
        Y_avg = np.mean(all_preds, axis=0)
        return np.argmax(Y_avg, axis=1)

# Train DIABLO
diablo = DIABLO(n_components=3, design=0.1)
X_list = [rna_scaled, prot_scaled, meth_scaled]
view_names = ['RNA-seq', 'Proteomics', 'Methylation']
diablo.fit(X_list, view_names, y_onehot)
block_scores = diablo.transform(X_list)

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_accs_diablo = []
for train, test in skf.split(rna_scaled, y):
    m = DIABLO(n_components=3)
    X_tr = [X[train] for X in X_list]
    X_te = [X[test] for X in X_list]
    m.fit(X_tr, view_names, y_onehot[train])
    y_pred = m.predict_class(X_te)
    cv_accs_diablo.append(balanced_accuracy_score(y[test], y_pred))

diablo_acc = np.mean(cv_accs_diablo)
diablo_std = np.std(cv_accs_diablo)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('DIABLO Multi-Block Integration', fontsize=13, fontweight='bold')

# Row 1: Component 1 vs 2 per block
for col, (name, scores) in enumerate(block_scores.items()):
    axes[0, col].scatter(scores[:, 0], scores[:, 1], c=point_colors, s=40, alpha=0.8)
    axes[0, col].set_xlabel('Comp 1'); axes[0, col].set_ylabel('Comp 2')
    axes[0, col].set_title(f'DIABLO - {name}')
handles = [mpatches.Patch(color=v, label=k) for k, v in class_colors.items()]
axes[0, -1].legend(handles=handles, fontsize=8, loc='lower right')

# Row 2, panel 1: CV accuracy comparison
per_view_accs = {name: cv_plsda(X, y_onehot, y)[0]*100 
                 for name, X in zip(view_names, X_list)}
per_view_accs['DIABLO\n(multi-block)'] = diablo_acc * 100

bar_colors_comp = ['steelblue', 'orange', 'green', 'red']
bars = axes[1, 0].bar(per_view_accs.keys(), per_view_accs.values(),
                       color=bar_colors_comp, alpha=0.8)
axes[1, 0].axhline(100/n_classes, color='black', linestyle='--', label='Random')
axes[1, 0].set_ylabel('Balanced Accuracy (%)')
axes[1, 0].set_title('5-CV Accuracy: Single View vs DIABLO')
axes[1, 0].legend()
for bar, val in zip(bars, per_view_accs.values()):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

# Row 2, panel 2: Between-block correlation
comp1_scores = np.array([block_scores[name][:, 0] for name in view_names])
corr_matrix = np.corrcoef(comp1_scores)
sns.heatmap(corr_matrix, ax=axes[1, 1], annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=view_names, yticklabels=view_names,
            vmin=-1, vmax=1)
axes[1, 1].set_title('Between-Block Correlation (Comp 1)')

# Row 2, panel 3: Confusion matrix of DIABLO
from sklearn.metrics import confusion_matrix
y_pred_final = diablo.predict_class(X_list)
cm = confusion_matrix(y, y_pred_final)
sns.heatmap(cm, ax=axes[1, 2], annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1, 2].set_xlabel('Predicted'); axes[1, 2].set_ylabel('True')
axes[1, 2].set_title('DIABLO Confusion Matrix')

plt.tight_layout()
plt.show()

print(f"\nDIABLO 5-CV balanced accuracy: {diablo_acc*100:.1f} +/- {diablo_std*100:.1f}%")
print(f"Improvement over best single-view: {(diablo_acc - max(v/100 for v in list(per_view_accs.values())[:3]))*100:.1f}%")


## 5. Model Performance Evaluation

### Cross-validation strategy for multi-omics

**Mfold CV** (recommended):
- Balanced folds (equal class representation)
- Multiple repeats (nrepeat=50 in mixOmics) to reduce variance
- Report: BER, AUC per fold and component

### Key metrics

| Metric | Interpretation |
|---|---|
| **BER** (balanced error rate) | 1 - balanced accuracy; accounts for class imbalance |
| **ROC-AUC** | Area under ROC curve; 1.0 = perfect, 0.5 = random |
| **Stability** | Proportion of CV folds where each feature is selected |

### Overfitting warning
With p >> n and multiple omics layers:
- Always use **nested CV** (outer loop for model evaluation, inner for hyperparameter tuning)
- Report 95% CI from repeated CV
- Be suspicious of BER < 5% — may indicate data leakage


In [ ]:
# ----- Comprehensive model evaluation -----
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

def evaluate_model(X_list_or_X, y, y_oh, model_type='diablo', n_comp=3, cv=5):
    """Full cross-validation with BER and AUC."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    metrics = {'acc': [], 'auc': []}
    
    for train, test in skf.split(X_list_or_X[0] if isinstance(X_list_or_X, list) else X_list_or_X, y):
        if model_type == 'diablo':
            X_tr = [X[train] for X in X_list_or_X]
            X_te = [X[test] for X in X_list_or_X]
            m = DIABLO(n_components=n_comp)
            m.fit(X_tr, view_names, y_oh[train])
            y_pred = m.predict_class(X_te)
            # For AUC: use average predictions
            y_proba = np.mean([m.block_models[name].predict(X_te[i]) 
                               for i, name in enumerate(view_names)], axis=0)
        else:
            X = X_list_or_X
            pls = PLSRegression(n_components=n_comp, scale=True)
            pls.fit(X[train], y_oh[train])
            y_pred = np.argmax(pls.predict(X[test]), axis=1)
            y_proba = pls.predict(X[test])
        
        y_proba = (y_proba - y_proba.min(axis=0)) / (y_proba.max(axis=0) - y_proba.min(axis=0) + 1e-8)
        y_proba = y_proba / y_proba.sum(axis=1, keepdims=True)
        
        metrics['acc'].append(balanced_accuracy_score(y[test], y_pred))
        try:
            auc = roc_auc_score(label_binarize(y[test], classes=[0,1,2]), y_proba, 
                                average='macro', multi_class='ovr')
            metrics['auc'].append(auc)
        except:
            metrics['auc'].append(np.nan)
    
    return {k: (np.nanmean(v), np.nanstd(v)) for k, v in metrics.items()}

# Compare all models
model_results = {}
model_results['PLS-DA RNA'] = evaluate_model(rna_scaled, y, y_onehot, 'plsda')
model_results['PLS-DA Prot'] = evaluate_model(prot_scaled, y, y_onehot, 'plsda')
model_results['PLS-DA Meth'] = evaluate_model(meth_scaled, y, y_onehot, 'plsda')
model_results['DIABLO'] = evaluate_model(X_list, y, y_onehot, 'diablo')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Model Performance Comparison', fontsize=12, fontweight='bold')

model_names = list(model_results.keys())
accs = [model_results[m]['acc'][0]*100 for m in model_names]
acc_errs = [model_results[m]['acc'][1]*100 for m in model_names]
aucs = [model_results[m]['auc'][0] for m in model_names]
auc_errs = [model_results[m]['auc'][1] for m in model_names]

bar_colors_ev = ['steelblue', 'steelblue', 'steelblue', 'red']
axes[0].bar(model_names, accs, yerr=acc_errs, color=bar_colors_ev, alpha=0.8, capsize=5)
axes[0].axhline(100/n_classes, color='black', linestyle='--', label='Random')
axes[0].set_ylabel('Balanced Accuracy (%)')
axes[0].set_title('5-fold CV Balanced Accuracy')
axes[0].legend()
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels(model_names, rotation=20, ha='right')

axes[1].bar(model_names, aucs, yerr=auc_errs, color=bar_colors_ev, alpha=0.8, capsize=5)
axes[1].axhline(0.5, color='black', linestyle='--', label='Random')
axes[1].set_ylabel('ROC-AUC (macro-average)')
axes[1].set_title('5-fold CV ROC-AUC')
axes[1].legend()
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels(model_names, rotation=20, ha='right')

plt.tight_layout()
plt.show()

print("\nModel comparison (5-fold CV):")
print(f"{'Model':<25} {'Acc%':>10} {'AUC':>10}")
for name, res in model_results.items():
    print(f"  {name:<23} {res['acc'][0]*100:>8.1f}  {res['auc'][0]:>8.3f}")


## 6. Summary

### Key concepts covered

1. **PLS framework**: maximizes X-Y covariance; ideal for high-dimensional omics
2. **PLS-DA**: supervised single-omics classification; components visualize class separation
3. **sPLS-DA**: adds sparsity (keepX); feature selection with bootstrap stability
4. **DIABLO**: multi-block supervised integration; design matrix controls between-block coupling
5. **Cross-validation**: BER + AUC as evaluation metrics; multiple repeats reduce variance

### Comparison: MOFA2 vs DIABLO

| Aspect | MOFA2 | DIABLO |
|---|---|---|
| Supervision | Unsupervised | Supervised (requires labels) |
| Output | Latent factors + variance explained | Predictive components + feature loadings |
| Missing views | Yes | No |
| Use case | Explore shared structure | Classify/biomarker discovery |
| Feature selection | ARD prior (automatic) | keepX (user-tuned) |

### Recommended workflow
1. **MOFA2** first: unsupervised exploration of shared structure
2. **sPLS-DA per layer**: assess discriminative power of each omics
3. **DIABLO**: supervised integration if classification is the goal
4. **Feature stability**: report bootstrap-stable features as biomarker candidates

### Next steps
- [Module 28: Network Biology](../28_Network_Biology/) — integrate DIABLO-selected genes into PPI networks


---[< Previous: MOFA2: Multi-Omics Factor Analysis](02_mofa2.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: PPI Network Construction & Analysis >](../28_Network_Biology/01_ppi_networks.ipynb)---